# Vígil.ia — Aprendizado ativo com humano no circuito (correção → reaprende)

3 fases:

1. **FASE 1 (auto):** o YOLO11x inspeciona o vídeo, dá um **#ID** a cada grão (rastreamento),
   fecha a classe por voto temporal e gera: recortes por grão, **folhas de contato** (pra revisar no celular)
   e uma **`revisao.csv`** com todos os grãos.
2. **FASE 2 (você):** abre a planilha, olha as folhas de contato e preenche `classe_corrigida`
   **só nas linhas erradas** (linha vazia = confirmado certo; `descartar` = não é grão/lixo).
3. **FASE 3 (auto):** cada correção do #ID se propaga pra **todos os frames** daquele grão →
   dataset revisado → re-treina o 11x → A/B pra provar que melhorou.

Uma correção sua vira dezenas de exemplos de treino. É *human-in-the-loop active learning*.

Pré-requisitos no Drive: `soja_yolo11x_v3.pt`, o vídeo a inspecionar, fotos reais (p/ Fase 3).

## 0. Setup e caminhos

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, glob, shutil, json
from google.colab import drive
drive.mount('/content/drive')

YOLOX_PT = '/content/drive/MyDrive/soja_yolo11x_v3.pt'
assert os.path.exists(YOLOX_PT), 'soja_yolo11x_v3.pt não encontrado no Drive!'

# Vídeo a inspecionar/corrigir (use um dos mistos; troque à vontade a cada rodada)
VIDEO_INSPECT = '/content/drive/MyDrive/teste_soja.mp4'
assert os.path.exists(VIDEO_INSPECT), f'vídeo não encontrado: {VIDEO_INSPECT}'

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']

# nome curto -> rótulo PT (só pra folha de contato ficar legível)
PT = {'broken': 'quebrado', 'immature': 'imaturo', 'intact': 'INTACTO',
      'skin-damaged': 'casca-dan', 'spotted': 'manchado'}

OUT = '/content/revisao_ativa'
CONF, IMGSZ, IOU = 0.35, 640, 0.5
SAVE_STRIDE = 4   # salva 1 a cada 4 frames p/ treino (evita quase-duplicatas)

# FASE 1 — Inspeção automática
Rastreia o vídeo, guarda a melhor foto (mais nítida) de cada grão, salva frames
amostrados p/ treino e monta a planilha de revisão.

In [ ]:
from collections import defaultdict, Counter
import cv2, numpy as np
from ultralytics import YOLO

shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(f'{OUT}/frames', exist_ok=True)
os.makedirs(f'{OUT}/graos', exist_ok=True)

model = YOLO(YOLOX_PT)
votes = defaultdict(Counter)   # id -> votos de classe (peso = confiança)
seen = Counter()               # id -> nº de frames
best = {}                      # id -> (nitidez, crop_bgr, conf)
frame_dets = {}                # frame_key -> [(tid, x1, y1, x2, y2)]

k = saved = 0
for r in model.track(source=VIDEO_INSPECT, imgsz=IMGSZ, iou=IOU, conf=CONF,
                     agnostic_nms=True, tracker='bytetrack.yaml',
                     stream=True, persist=True, verbose=False):
    frame = r.orig_img
    dets = []
    if r.boxes.id is not None:
        for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                                    r.boxes.id.int().tolist(),
                                    r.boxes.cls.int().tolist(),
                                    r.boxes.conf.tolist()):
            x1, y1, x2, y2 = [int(v) for v in xyxy]
            votes[tid][NAMES[c]] += cf
            seen[tid] += 1
            dets.append((tid, x1, y1, x2, y2))
            crop = frame[max(0, y1):y2, max(0, x1):x2]
            if crop.size:
                sh = cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
                if tid not in best or sh > best[tid][0]:
                    best[tid] = (sh, crop.copy(), cf)
    if k % SAVE_STRIDE == 0 and dets:
        cv2.imwrite(f'{OUT}/frames/frame_{k:05d}.jpg', frame)
        frame_dets[k] = dets
        saved += 1
    k += 1

verdict = {tid: cnt.most_common(1)[0][0] for tid, cnt in votes.items()}
conf_of = {tid: round(best[tid][2], 3) if tid in best else 0.0 for tid in verdict}

# persiste tudo (sobrevive a queda de sessão)
json.dump({'frame_dets': {str(a): b for a, b in frame_dets.items()},
           'verdict': verdict, 'seen': dict(seen), 'conf_of': conf_of},
          open(f'{OUT}/meta.json', 'w'))

dist = Counter(verdict.values())
print(f'{k} frames | {saved} salvos p/ treino | {len(verdict)} grãos rastreados')
print('distribuição de classes:', dict(dist))

In [ ]:
import csv
import matplotlib.pyplot as plt

# 1) recorte de review por grão + 2) planilha + 3) folhas de contato
for tid in verdict:
    if tid in best:
        cv2.imwrite(f'{OUT}/graos/{tid:04d}_{verdict[tid]}.jpg', best[tid][1])

with open(f'{OUT}/revisao.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'classe_prevista', 'confianca', 'n_frames', 'classe_corrigida'])
    for tid in sorted(verdict):
        w.writerow([tid, verdict[tid], conf_of[tid], seen[tid], ''])

ids = sorted(best)
PER, COLS = 24, 6
for page in range(0, len(ids), PER):
    chunk = ids[page:page + PER]
    rows_ = (len(chunk) + COLS - 1) // COLS
    plt.figure(figsize=(COLS * 2.2, rows_ * 2.5))
    for i, tid in enumerate(chunk):
        ax = plt.subplot(rows_, COLS, i + 1)
        ax.imshow(cv2.cvtColor(best[tid][1], cv2.COLOR_BGR2RGB))
        cls = verdict[tid]
        ax.set_title(f'#{tid}  {PT[cls]}\n{conf_of[tid]:.2f}', fontsize=8,
                     color=('green' if cls == 'intact' else 'firebrick'))
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{OUT}/folha_contato_{page // PER:02d}.png', dpi=110)
    plt.show()

# backup no Drive (frames p/ treino + meta) e a planilha SOLTA p/ você editar
shutil.make_archive('/content/revisao_ativa', 'zip', OUT)
!cp /content/revisao_ativa.zip /content/drive/MyDrive/
!cp {OUT}/revisao.csv /content/drive/MyDrive/revisao.csv
print('\nNo Drive: revisao_ativa.zip (frames+recortes) e revisao.csv (edite este)')

# FASE 2 — Correção (você, ~10 min)

1. Abra **`revisao.csv`** no Drive (Google Sheets edita direto).
2. Olhe as **folhas de contato** acima (ou os recortes em `graos/`) — cada grão tem seu **#ID**.
3. Na coluna **`classe_corrigida`**, preencha **só onde o modelo errou**:
   - deixe **vazio** = a previsão está certa (isso também vira rótulo verificado!)
   - escreva a classe certa = correção. Aceita PT ou inglês: `manchado`/`spotted`,
     `quebrado`/`broken`, `imaturo`/`immature`, `casca danificada`/`skin-damaged`, `intacto`/`intact`
   - escreva **`descartar`** = não é grão (sujeira, reflexo) → sai do treino
4. Salve o CSV **no mesmo lugar do Drive** (`MyDrive/revisao.csv`) e rode a Fase 3.

> Dica: revise primeiro os de **confiança baixa** (`confianca` < 0,6) — é onde ele mais erra.

# FASE 3 — Reaprendizado
Aplica suas correções a todos os frames de cada grão, junta com o dataset v3 e
re-treina o 11x a partir do campeão. Batch 16 (A100).

In [ ]:
# --- dataset v3 (reconstrói se a sessão for nova) — necessário p/ merge + val ---
import hashlib, unicodedata, yaml
import cv2, numpy as np
RNG = np.random.default_rng(42)
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    kk = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((kk, kk), np.float32)
    kernel[kk // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((kk / 2 - 0.5, kk / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (kk, kk))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict as dd
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = dd(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    for sp in ('train', 'val'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    for i, (path, cls, sp) in enumerate(items):
        img = cv2.imread(path)
        if img is None:
            continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb), [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
    for j in range(n_synth):
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
    return f'{out_dir}/data.yaml'

DS = '/content/soja_det_v3'
if not os.path.exists(f'{DS}/images/val'):
    print('construindo dataset v3 (10-15 min)…')
    build_v3(collect_real(REAL_SRCS), DS)
print('dataset v3 pronto')

In [ ]:
# --- lê a planilha corrigida e propaga cada correção a TODOS os frames do grão ---
# recarrega meta (se a sessão caiu, restaura do zip do Drive)
import csv
if not os.path.exists(f'{OUT}/meta.json'):
    !unzip -q -o /content/drive/MyDrive/revisao_ativa.zip -d {OUT}
meta = json.load(open(f'{OUT}/meta.json'))
frame_dets = {int(a): b for a, b in meta['frame_dets'].items()}

def resolve(corr, prevista):
    v = corr.strip().lower()
    if v == '':
        return prevista
    if v in ('descartar', 'x', 'drop', 'lixo', 'sujeira'):
        return None
    c = class_of(v)
    if c is not None:
        return NAMES[c]
    if v in NAMES:
        return v
    raise ValueError(f'classe_corrigida inválida: {corr!r} — use PT/inglês das 5 classes ou "descartar"')

final_cls, drop = {}, set()
n_corr = 0
for row in csv.DictReader(open('/content/drive/MyDrive/revisao.csv')):
    tid = int(row['id'])
    fc = resolve(row.get('classe_corrigida', ''), row['classe_prevista'])
    if row.get('classe_corrigida', '').strip():
        n_corr += 1
    if fc is None:
        drop.add(tid)
    else:
        final_cls[tid] = fc
print(f'{n_corr} linhas com anotação sua | {len(drop)} descartados | {len(final_cls)} grãos p/ treino')

# escreve frames revisados (oversample: correção pesa mais no treino)
REV = '/content/soja_rev'
shutil.rmtree(REV, ignore_errors=True)
os.makedirs(f'{REV}/images/train', exist_ok=True)
os.makedirs(f'{REV}/labels/train', exist_ok=True)
IDX = {n: i for i, n in enumerate(NAMES)}
REV_REPEAT = 3
written = 0
for kf, dets in frame_dets.items():
    fp = f'{OUT}/frames/frame_{kf:05d}.jpg'
    if not os.path.exists(fp):
        continue
    img = cv2.imread(fp)
    H, W = img.shape[:2]
    lines = []
    for tid, x1, y1, x2, y2 in dets:
        if tid in drop or tid not in final_cls:
            continue
        cx = ((x1 + x2) / 2) / W
        cy = ((y1 + y2) / 2) / H
        w = (x2 - x1) / W
        h = (y2 - y1) / H
        lines.append(f'{IDX[final_cls[tid]]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    if not lines:
        continue
    for j in range(REV_REPEAT):
        stem = f'frame_{kf:05d}_r{j}'
        cv2.imwrite(f'{REV}/images/train/{stem}.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{REV}/labels/train/{stem}.txt', 'w').write('\n'.join(lines))
    written += 1
print(f'{written} frames revisados escritos (x{REV_REPEAT} = {written*REV_REPEAT} exemplos)')

ACTIVE_YAML = '/content/soja_active.yaml'
yaml.safe_dump({'train': [f'{DS}/images/train', f'{REV}/images/train'],
                'val': f'{DS}/images/val',
                'names': {i: n for i, n in enumerate(NAMES)}},
               open(ACTIVE_YAML, 'w'), sort_keys=False, allow_unicode=True)
print('dataset ativo:', ACTIVE_YAML)

In [ ]:
# --- re-treina o 11x a partir do CAMPEÃO, com as correções embutidas ---
YOLO(YOLOX_PT).train(
    data=ACTIVE_YAML, epochs=40, imgsz=640, batch=16, device=0, seed=42,
    optimizer='AdamW', lr0=5e-4, patience=15, close_mosaic=8,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_active', name='yolo11x_active', exist_ok=True,
)
!cp /content/runs/detect/runs_active/yolo11x_active/weights/best.pt /content/drive/MyDrive/soja_yolo11x_active.pt
print('backup ok: soja_yolo11x_active.pt')

In [ ]:
# --- A/B: campeão vs corrigido no mesmo split de val ---
print('mAP no val v3 — campeão vs corrigido pelo humano:\n')
for tag, w in [('yolo11x v3 (campeão)', YOLOX_PT),
               ('yolo11x active (corrigido)', '/content/runs/detect/runs_active/yolo11x_active/weights/best.pt')]:
    r = YOLO(w).val(data=ACTIVE_YAML, split='val', imgsz=640, device=0, verbose=False)
    print(f'{tag:28s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')
print('\nJuiz de verdade: rode o modelo corrigido no MESMO vídeo (Fase 1) e veja se os')
print('grãos que você corrigiu agora saem certos — e repita o ciclo com o próximo vídeo.')

## Ciclo de melhoria contínua

Cada rodada: inspeciona um vídeo novo → corrige os erros na planilha → reaprende.
O `soja_yolo11x_active.pt` vira o novo campeão da próxima rodada (troque `YOLOX_PT` por ele).

**Cuidado registrado:** revise sempre num vídeo — o modelo aprende com as próprias
previsões confirmadas, então erros seus na planilha viram erros dele. A folha de contato
existe pra você conferir com cuidado, especialmente os de confiança baixa.